In [6]:
# Webスクレイピングに必要なライブラリをインポート
import time
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse


In [7]:
# アクセス先（課題の対象）
start_url = "https://www.musashino-u.ac.jp/"

# リクエストの共通ヘッダ（先生の例に合わせて）
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
}

# サーバ負荷軽減のための待機秒（課題要件）
SLEEP_SEC = 1.0

# クロール上限（安全のため適度に制限）
MAX_PAGES = 150

# 同一ドメイン判定用
root_netloc = urlparse(start_url).netloc.lower()

def is_same_domain(u):
    netloc = urlparse(u).netloc.lower()
    return netloc.endswith("." + root_netloc) or netloc == root_netloc

# 画像・PDF等の非HTMLを除外（拡張子フィルタ）
SKIP_EXTS = {".png",".jpg",".jpeg",".gif",".svg",".pdf",".zip",".css",".js",".mp4",".mov",".webm",".ico"}


In [8]:
from collections import deque
from pathlib import Path

to_visit = deque([start_url])
visited = set()
url_title_dict = {}  # 課題要件： key=URL, value=<title>文字列

while to_visit and len(visited) < MAX_PAGES:
    url = to_visit.popleft()
    if url in visited:
        continue
    if not is_same_domain(url):
        continue
    # 拡張子で非HTMLをスキップ
    if Path(urlparse(url).path).suffix.lower() in SKIP_EXTS:
        visited.add(url)
        continue

    try:
        res = requests.get(url, headers=headers, timeout=15)
    except requests.RequestException:
        visited.add(url)
        time.sleep(SLEEP_SEC)
        continue

    # 先生の例に合わせてエンコーディングを設定
    res.encoding = res.apparent_encoding

    # HTML以外（例：application/pdf等）は除外
    ctype = res.headers.get("Content-Type","")
    if res.status_code != 200 or "text/html" not in ctype:
        visited.add(url)
        time.sleep(SLEEP_SEC)
        continue

    # HTML解析
    soup = BeautifulSoup(res.text, "html.parser")

    # <title> を取り出して辞書に保存（無い場合は空文字）
    title = soup.title.string.strip() if soup.title and soup.title.string else ""
    url_title_dict[url] = title
    visited.add(url)

    # 同一ドメイン内のリンクを収集（コメント内のaはBS4が通常拾わないので要件クリア）
    for a in soup.find_all("a", href=True):
        abs_url = urljoin(url, a["href"]).split("#")[0]  # フラグメント除去
        if (
            abs_url not in visited
            and is_same_domain(abs_url)
            and Path(urlparse(abs_url).path).suffix.lower() not in SKIP_EXTS
        ):
            to_visit.append(abs_url)

    # サーバ負荷軽減（課題要件のtime.sleep）
    time.sleep(SLEEP_SEC)


In [9]:
# 辞書型変数をprint()で表示
print(url_title_dict)

# 見やすい参考出力（任意）
print("\n---- URLs & Titles (sample) ----")
count = 0
for u, t in url_title_dict.items():
    print(u, "=>", t)
    count += 1
    if count >= 50:  # 多すぎる場合は一部だけ
        break
print(f"\nTotal pages collected: {len(url_title_dict)}")


{'https://www.musashino-u.ac.jp/': '武蔵野大学', 'https://www.musashino-u.ac.jp/access.html': '交通アクセス | 武蔵野大学', 'https://www.musashino-u.ac.jp/admission/request.html': '資料請求 | 入試情報 | 武蔵野大学', 'https://www.musashino-u.ac.jp/contact.html': 'お問い合わせ | 武蔵野大学', 'https://www.musashino-u.ac.jp/prospective-students.html': '武蔵野大学で学びたい方 | 武蔵野大学', 'https://www.musashino-u.ac.jp/students.html': '在学生の方 | 武蔵野大学', 'https://www.musashino-u.ac.jp/alumni.html': '卒業生の方 | 武蔵野大学', 'https://www.musashino-u.ac.jp/parents.html': '保護者の方 | 武蔵野大学', 'https://www.musashino-u.ac.jp/business.html': '企業・研究者の方 | 武蔵野大学', 'https://www.musashino-u.ac.jp/guide/': '大学案内 | 武蔵野大学', 'https://www.musashino-u.ac.jp/guide/profile/': '大学紹介 | 大学案内 | 武蔵野大学', 'https://www.musashino-u.ac.jp/guide/activities/': '大学の取り組み | 大学案内 | 武蔵野大学', 'https://www.musashino-u.ac.jp/guide/campus/': 'キャンパス | 大学案内 | 武蔵野大学', 'https://www.musashino-u.ac.jp/guide/facility/': '附置機関・センター・附属施設 | 大学案内 | 武蔵野大学', 'https://www.musashino-u.ac.jp/guide/information/': '情報

In [10]:
# 注意点メモ（MarkdownセルでもOK）
print(
"""【注意点】
- アクセス間隔：time.sleep(1.0) を必ず入れてサーバ負荷を軽減。
- 対象範囲：同一ドメイン（*.musashino-u.ac.jp）内のページのみを探索。
- 画像・PDF等は除外：.png/.jpg/.pdf などの拡張子URLはスキップ。
- コメント内のリンク：BeautifulSoupのfind_all('a')は通常コメント内を解析しないため、
  課題の「コメントアウトされていないもののみ」を満たす。
- 出力形式：辞書型 {URL: <title文字列>} を print() で表示。
"""
)


【注意点】
- アクセス間隔：time.sleep(1.0) を必ず入れてサーバ負荷を軽減。
- 対象範囲：同一ドメイン（*.musashino-u.ac.jp）内のページのみを探索。
- 画像・PDF等は除外：.png/.jpg/.pdf などの拡張子URLはスキップ。
- コメント内のリンク：BeautifulSoupのfind_all('a')は通常コメント内を解析しないため、
  課題の「コメントアウトされていないもののみ」を満たす。
- 出力形式：辞書型 {URL: <title文字列>} を print() で表示。

